# MolSanity — full-scale audit run on a free Colab GPU

Runs `configs/full.yaml`: **57 cells x 2 splits x 3 seeds = 342 cell-runs**,
150 epochs, 50 IG steps, up to 200 audited molecules per cell, across every
reachable dataset x backbone x attributor.

### What changed since the last run

This sweep is not an increment on the previous one. Five things changed, and
together they mean **every number moves**:

| change | why | effect |
|---|---|---|
| test fold 0.1 → **0.3**, SynthMotifs 200 → **1000** graphs | n was 20 per ground-truth arm because that *was* the test split of a 188-graph dataset, not a budget choice | n per arm ≈ 56 / 200 / 200 / 49 |
| `seeds: [0, 1, 2]` | one deterministic split at one initialisation could not distinguish a real effect from run-to-run noise | new `SEED_VARIANCE.md` |
| **BA-2Motifs** node labels recovered | PyG's loader ships none, so it contributed no ground-truth cells; they are recoverable from its node ordering | third exact-GT arm |
| **ShapeGGen** wired via k-hop subgraphs | it is node classification, so each labelled node's computation graph becomes one instance | fourth exact-GT arm; closes the 86/88 grid |
| **SubgraphX** wrapped from DIG | the "no wheel exists" blocker was wrong — the extensions build from source | the perturbation family is no longer GNNExplainer alone |

Because the split is part of the checkpoint hash, **all models retrain**. That
is intended: the old weights were fitted on a different training set. Nothing in
`artifacts/` is invalid, it simply no longer matches.

If the full budget is too much, set `CONFIG = 'configs/groundtruth.yaml'` in
step 6b: 35 cells, 210 cell-runs, covering every dataset where attribution
*correctness* is measurable at all. That is where the selection experiment, the
seed variance and the shift contrast live.

### How to run
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough — the models
   are ~43k params, <2 GB VRAM).
2. **The repo is private** → paste a GitHub token into the clone cell (step 4).
   Without it the clone fails and nothing downstream can work.
3. **Runtime → Run all.** Step 5b builds `torch_sparse` from source; allow
   10–15 minutes for it. If it fails, SubgraphX and ShapeGGen cells are skipped
   and logged, and the rest of the sweep is unaffected.

Steps 2 (keep-alive) and 3 (Drive) guard a long run against Colab disconnects.
The pipeline is **resumable**: stage `.done` markers are reused, so after a drop
you re-run the setup cells plus *Run the sweep* and it continues.

## 1. Verify the GPU

In [ ]:
import subprocess

import torch

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(smi or 'nvidia-smi not found')
if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU detected. Set Runtime > Change runtime type > T4 GPU, then Run all.'
    )
print('CUDA:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Keep the session alive

Colab disconnects an *idle* runtime after ~90 minutes. This clicks the connect
button on a timer so a long unattended run is not killed for inactivity.

**What it does not do** — the honest limits:

- it does **not** survive closing the browser tab (the JS dies with the page);
- it does **not** extend Colab's hard max-runtime cap (~12 h on the free tier);
- it does **not** stop Colab reclaiming a GPU when demand is high.

So treat it as insurance against *idle* timeout only. The durable protection is
step 3 (persist to Drive) plus the pipeline's resumability.

In [ ]:
from IPython.display import Javascript, display

_js = '\n'.join([
    "  function keepAlive() {",
    "    const btn = document.querySelector('colab-connect-button');",
    "    if (btn) { btn.click(); console.log('keep-alive', new Date().toISOString()); }",
    "  }",
    "  if (window._molsanityKeepAlive) { clearInterval(window._molsanityKeepAlive); }",
    "  window._molsanityKeepAlive = setInterval(keepAlive, 60 * 1000);",
])
display(Javascript(_js))
print('Keep-alive armed — pings once a minute while this tab stays open.')

## 3. (Recommended) Persist the run state to Google Drive

Only `artifacts/` needs to survive a disconnect: it holds the trained
checkpoints and the stage `.done` markers that make the sweep resumable.

The **repo itself is deliberately kept on local disk**, not on Drive. Drive is a
FUSE mount whose file caching breaks git's bookkeeping — a repo cloned there
fails to update with `fatal: shallow file has changed since we read it`, which
is how a re-run once ended up silently running old code. Cloning locally and
symlinking `artifacts/` onto Drive gives fast, reliable git *and* durable run
state.

Set `USE_DRIVE = False` for a throwaway run.


In [ ]:
USE_DRIVE = True

import os

WORKDIR = '/content'          # repo lives here: local disk, fast and git-safe
DRIVE_DIR = None              # persistent run state (artifacts/) lives here

if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount('/content/drive')
        DRIVE_DIR = '/content/drive/MyDrive/molsanity_runs'
        os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as exc:
        print('Drive unavailable — run state will NOT survive a disconnect:', exc)

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print('working directory:', os.getcwd())
print('persistent state :', DRIVE_DIR or '(none — ephemeral)')


## 4. Clone the repo (fresh every session)

`Kar488/molsanity` is **private**, so a GitHub personal access token with `repo`
scope is required — create one at <https://github.com/settings/tokens>.

The clone is **full, not shallow, and to local disk**, and it is re-made every
session. That is deliberate: it makes it impossible to run stale code, and it
avoids the shallow-fetch failure that Drive-hosted clones hit. The repo is only
~20 MB, so this costs seconds.

`artifacts/` is then symlinked onto Drive, so checkpoints and `.done` markers
persist across sessions and the sweep resumes. If a previous run left artifacts
inside a Drive-hosted clone, they are migrated automatically — nothing is lost.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

GITHUB_TOKEN = ''  # <- REQUIRED: this repo is private
OWNER, REPO = 'Kar488', 'molsanity'
BRANCH = 'main'   # <- point at the branch carrying the fixes if not yet merged

if not GITHUB_TOKEN:
    raise SystemExit(
        'GITHUB_TOKEN is empty and the repo is private, so the clone would fail. '
        'Create a token (repo scope) at https://github.com/settings/tokens, '
        'paste it above, and re-run this cell.'
    )

url = f'https://{GITHUB_TOKEN}@github.com/{OWNER}/{REPO}.git'
repo_path = Path(WORKDIR) / REPO


def git(*args, cwd=None):
    r = subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit('git ' + args[0] + ' failed:\n'
                         + (r.stderr or r.stdout).replace(GITHUB_TOKEN, '***'))
    return r


# 1. Fresh clone every session — never an in-place update, so stale code is
#    impossible. rmtree does not follow the artifacts symlink, so Drive state
#    is safe.
if repo_path.exists():
    link = repo_path / 'artifacts'
    if link.is_symlink():
        link.unlink()
    shutil.rmtree(repo_path)
git('clone', '--branch', BRANCH, url, str(repo_path))
os.chdir(repo_path)

for marker in ('molsanity', 'configs/full.yaml'):
    if not os.path.exists(marker):
        raise SystemExit(f'Not at the repo root — missing {marker!r} in {os.getcwd()}')
REPO_DIR = os.getcwd()

# 2. Point artifacts/ at durable storage, migrating any earlier layout.
if DRIVE_DIR:
    drive_artifacts = Path(DRIVE_DIR) / 'artifacts'
    legacy = Path(DRIVE_DIR) / REPO / 'artifacts'   # the old Drive-hosted clone
    if not drive_artifacts.exists() and legacy.exists():
        shutil.move(str(legacy), str(drive_artifacts))
        print('migrated run state from the previous Drive-hosted clone')
    if not drive_artifacts.exists():
        # First use: seed it with whatever the repo ships (committed checkpoints)
        # so nothing that was tracked in git is lost.
        shutil.copytree(repo_path / 'artifacts', drive_artifacts)
        print('seeded persistent artifacts/ from the repo')
    shutil.rmtree(repo_path / 'artifacts', ignore_errors=True)
    (repo_path / 'artifacts').symlink_to(drive_artifacts, target_is_directory=True)

markers = list(Path('artifacts').glob('cell_*/.done'))
ckpts = list(Path('artifacts/checkpoints').glob('*.pt'))
print('repo    :', REPO_DIR)
print('branch  :', git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip(),
      '@', git('rev-parse', '--short', 'HEAD').stdout.strip())
print('artifacts ->', os.path.realpath('artifacts'))
print(f'resume state: {len(markers)} completed cells, {len(ckpts)} checkpoints')
subprocess.run(['git', 'log', '--oneline', '-1'])


## 5. Install dependencies

Colab ships a CUDA build of PyTorch; PyG (2.5+), RDKit, and Captum are
pure-Python wheels that need no compiled extensions. PyTDC (for DILI / hERG /
Tox21) is installed with a minimal footprint so it does not fight Colab's pinned
numpy/pandas.

In [ ]:
%pip install -q torch-geometric rdkit captum pyyaml
# Minimal PyTDC: skip its heavy optional deps (transformers/scanpy/...).
%pip install -q --no-deps PyTDC huggingface_hub httpx fuzzywuzzy
print('core deps installed')


### 5b. SubgraphX (DIG) and ShapeGGen (GraphXAI) — optional

Both are optional. **If either fails or you skip it, the sweep still runs**; the
affected cells are skipped and logged.

`torch_scatter`/`torch_sparse` have no wheel on PyPI, so `pip install` falls
back to compiling them from source. On a Colab GPU runtime that means building
CUDA kernels on 2 vCPUs and can take **well over an hour**. The cell below
avoids that by installing from PyG's prebuilt wheel index first, matched to the
runtime's exact torch and CUDA versions, which takes seconds. It only falls back
to a source build if no matching wheel exists.

Set `INSTALL_SUBGRAPHX = False` to skip it entirely and start the sweep now;
SubgraphX cells will be logged as skipped and everything else is unaffected.

In [ ]:
INSTALL_SUBGRAPHX = True   # False -> skip, sweep runs without SubgraphX cells
INSTALL_SHAPEGGEN = True

import os
import subprocess
import sys

import torch

if INSTALL_SUBGRAPHX:
    # PyG publishes prebuilt wheels keyed by exact torch + CUDA build. Using
    # them turns an hour-long source compile into a few seconds. If the runtime
    # has no matching wheel we say so and fall back, rather than silently
    # starting a build that looks like a hang.
    cuda = torch.version.cuda
    tag = f"{torch.__version__.split('+')[0]}+{'cu' + cuda.replace('.', '') if cuda else 'cpu'}"
    index = f'https://data.pyg.org/whl/torch-{tag}.html'
    print(f'torch {torch.__version__} (cuda {cuda}) -> {index}')

    rc = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch-scatter', 'torch-sparse', '-f', index],
        capture_output=True, text=True).returncode

    if rc != 0:
        print('  no prebuilt wheel for this runtime; building from source.')
        print('  This is the slow path (30-90 min on Colab). Interrupt and set')
        print('  INSTALL_SUBGRAPHX = False if you would rather start the sweep.')
        # Use every core the runtime has, instead of the default single job.
        os.environ['MAX_JOBS'] = str(os.cpu_count() or 2)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch-scatter', 'torch-sparse'], check=False)

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'dive-into-graphs'], check=False)

if INSTALL_SHAPEGGEN:
    # GraphXAI's published wheel packages only the top-level module, so it
    # imports but is unusable. Install from a source checkout.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipdb'],
                   check=False)
    GRAPHXAI_DIR = '/content/GraphXAI'
    if not os.path.isdir(GRAPHXAI_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/mims-harvard/GraphXAI.git',
                        GRAPHXAI_DIR], check=False)
    if GRAPHXAI_DIR not in sys.path:
        sys.path.insert(0, GRAPHXAI_DIR)
    # run_all is launched as a subprocess, so pass the path via the environment.
    os.environ['PYTHONPATH'] = GRAPHXAI_DIR + os.pathsep + os.environ.get('PYTHONPATH', '')

print()
for name, what in [('torch_scatter', 'SubgraphX'), ('torch_sparse', 'SubgraphX'),
                   ('dig', 'SubgraphX'), ('graphxai', 'ShapeGGen')]:
    try:
        __import__(name)
        print(f'  OK       {name:16s} -> {what} enabled')
    except Exception as exc:
        print(f'  MISSING  {name:16s} -> {what} cells skipped and logged '
              f'({type(exc).__name__})')


In [ ]:
%pip install -q -e .
print('molsanity installed (editable)')

## 6. Smoke check — imports plus a real dataset load

In [ ]:
import molsanity  # noqa: F401
from molsanity.data.datasets import load_dataset

ld = load_dataset('MUTAG')
print('MUTAG:', len(ld.dataset), 'graphs — pipeline ready')

## 6b. Resume preflight — what will actually re-run

Before spending GPU hours, check what the sweep will skip and what it will redo.

A cell is skipped only if `artifacts/cell_<id>/.done` exists **and** its config
hash still matches. That marker is written *after* the cell succeeds, so a cell
that failed last time has no marker and re-runs automatically — there is no
"failed checkpoint" to clear.

**Expect everything to re-run this time.** The evaluation split changed from
0.8/0.1/0.1 to 0.6/0.1/0.3, which is what lifts the ground-truth arms off
n=20. The split is part of both the stage hash and the checkpoint hash, so
every cell is stale and every model retrains. That is intended, not a cache
miss: the old weights were fitted on a different training set.

The config also now carries `seeds: [0, 1, 2]`, so each cell runs three times
with independent splits and initialisations.

In [ ]:
from pathlib import Path

import yaml

from molsanity.utils import hash_config

CONFIG = 'configs/full.yaml'   # or 'configs/groundtruth.yaml' for the GT arms only

cfg = yaml.safe_load(Path(CONFIG).read_text())
splits = [cfg['split']['kind']] + list(cfg.get('extra_splits') or [])
seeds = [int(s) for s in (cfg.get('seeds') or [cfg.get('seed', 0)])]
multi_seed = len(seeds) > 1

done, stale, todo = [], [], []
for cell in cfg['cells']:
    for sp in (cell.get('splits') or ([cell['split']] if cell.get('split') else splits)):
        for seed in seeds:
            base = f"{cell['dataset']}__{cell['backbone']}__{cell['attributor']}__{sp}"
            cid = f'{base}__seed{seed}' if multi_seed else base
            stage_cfg = {'cell': cell, 'split': sp, 'budget': cfg.get('budget'),
                         'model': cfg['model'], 'train': cfg['train'], 'seed': seed}
            marker = Path('artifacts') / f'cell_{cid}' / '.done'
            if not marker.exists():
                todo.append(cid)
            elif marker.read_text().strip() == hash_config(stage_cfg):
                done.append(cid)
            else:
                stale.append(cid)

print(f'config    : {CONFIG}')
print(f'seeds     : {seeds}')
print(f'cell-runs : {len(cfg["cells"])} cells x {len(splits)} splits x '
      f'{len(seeds)} seeds = {len(done) + len(stale) + len(todo)}')
print()
print(f'cached, will be skipped    : {len(done)}')
print(f'config changed, will re-run: {len(stale)}')
print(f'no marker, will run        : {len(todo)}')

ck = sorted(Path('artifacts/checkpoints').glob('*/*.pt'))
print(f'checkpoints on disk        : {len(ck)} '
      f'(stale ones are ignored, not reused, once the split changes)')


## 7. Run the sweep

`configs/full.yaml` is **58 cells x 2 splits x 3 seeds = 348 cell-runs**, and
SubgraphX runs a Monte-Carlo tree search per molecule, so it is the slowest
attributor by a wide margin. `configs/groundtruth.yaml` is the same thing
restricted to the four datasets where attribution *correctness* is measurable
(216 cell-runs); the selection experiment, the seed variance and the shift
contrast all live there.

The run is resumable: re-execute this cell after a disconnect and it picks up
from the last completed cell.

Because the split changed, the previous `RESULTS.md` describes models trained on
a different training set. The cell below moves it aside rather than letting old
and new rows sit in one table.

In [ ]:
import os
import shutil
import subprocess
import time
from datetime import datetime
from pathlib import Path

os.chdir(REPO_DIR)  # guard: never run the sweep from the wrong directory

# The split changed, so previously reported rows are not comparable to the new
# ones. Archive rather than delete, so nothing is lost.
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
for name in ('RESULTS.md', 'BENCHMARK.md', 'BENCHMARK_GT.md', 'SEED_VARIANCE.md'):
    p = Path(name)
    if p.exists():
        shutil.move(str(p), f'{p.stem}.pre-{stamp}{p.suffix}')
        print(f'archived {name} -> {p.stem}.pre-{stamp}{p.suffix}')

t0 = time.time()
proc = subprocess.run(
    ['python', '-m', 'molsanity.run_all', '--config', CONFIG],
    text=True, env={**os.environ},
)
print(f'\nfinished in {(time.time() - t0) / 60:.1f} min (exit {proc.returncode})')


## 8. Results

`SEED_VARIANCE.md` is the new one and the one to read first. It gives the
across-seed mean and standard deviation for every cell. Compare that spread
against the effect sizes in `RESULTS.md`: an effect smaller than the seed
spread is not evidence, and now you can tell which is which.

In [ ]:
from pathlib import Path

for name in ['SEED_VARIANCE.md', 'RESULTS.md', 'BENCHMARK.md', 'BENCHMARK_GT.md']:
    p = Path(name)
    if p.exists():
        print('=' * 80, '\n', name, '\n', '=' * 80)
        print(p.read_text())


### 8b. Did the ground-truth arms actually get bigger?

The point of the split change was to lift n off 20. This checks it happened,
and reports the seed spread on the metric the paper's headline rests on.

In [ ]:
import re
from pathlib import Path

rows = []
for line in Path('RESULTS.md').read_text().splitlines():
    if line.startswith('|') and not line.startswith('| ---') and 'dataset' not in line:
        cells = [c.strip() for c in line.strip('|').split('|')]
        if len(cells) >= 8:
            rows.append(cells)

gt_sets = {'MUTAG', 'SynthMotifs', 'SynthMotifsXL', 'BA-2Motifs', 'ShapeGGen'}
per_ds = {}
for r in rows:
    if r[0] in gt_sets:
        try:
            per_ds.setdefault(r[0], set()).add(int(r[5]))   # n_mol column
        except ValueError:
            pass

print('audited molecules per ground-truth arm (was 20 everywhere):')
for ds in sorted(per_ds):
    ns = sorted(per_ds[ds])
    print(f'  {ds:16s} n = {ns}')

sv = Path('SEED_VARIANCE.md')
if sv.exists():
    body = sv.read_text()
    m = re.search(r'## GT AUROC(.*?)(?=\n## |\Z)', body, re.S)
    print()
    print(m.group(0)[:2000] if m else 'no GT AUROC section (single-seed run?)')


## 9. Publish the run into `results/`

The sweep writes its reports to the repo root (`RESULTS.md`, `BENCHMARK.md`, …)
because those paths are relative to the working directory. That is fine as a
working location, but the *committed* record of a run lives in `results/`, which
is what the paper build (`paper/figs/msdata.py`) reads.

This step copies the run outputs there so there is one place to commit and no
ambiguity about which numbers are current. Checkpoint `.pt` files are excluded —
they are large and git-ignored; `artifacts/checkpoints/MANIFEST.*` records what
was trained.


In [ ]:
import shutil
from pathlib import Path

REPORTS = ['RESULTS.md', 'BENCHMARK.md', 'BENCHMARK_GT.md', 'BENCHMARK_GT.json',
           'PROGRESS.md', 'LIMITATIONS.md']
TREES = ['figures', 'logs', 'artifacts/audit']

out = Path('results')
out.mkdir(exist_ok=True)
published = []

for name in REPORTS:
    if Path(name).exists():
        shutil.copy2(name, out / name)
        published.append(name)

for rel in TREES:
    src = Path(rel)
    if src.is_dir():
        dest = out / rel
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        published.append(rel + '/')

(out / 'artifacts/checkpoints').mkdir(parents=True, exist_ok=True)
if Path('artifacts/run_manifest.json').exists():
    shutil.copy2('artifacts/run_manifest.json', out / 'artifacts/run_manifest.json')
    published.append('artifacts/run_manifest.json')
for m in Path('artifacts/checkpoints').glob('MANIFEST.*'):
    shutil.copy2(m, out / 'artifacts/checkpoints' / m.name)
    published.append('artifacts/checkpoints/' + m.name)

# Trained weights: archived alongside the results so a clone answers "can you
# share your checkpoints?" without a Drive link. GitHub refuses files over
# 100 MB and warns over 50 MB, so anything that large is reported and skipped
# rather than silently breaking the push.
MAX_FILE_MB, MAX_TOTAL_MB = 45, 400
weights = sorted(Path('artifacts/checkpoints').rglob('*.pt'))
copied_mb, skipped = 0.0, []
for w in weights:
    mb = w.stat().st_size / 1e6
    if mb > MAX_FILE_MB or copied_mb + mb > MAX_TOTAL_MB:
        skipped.append((str(w), mb))
        continue
    dest = out / w
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(w, dest)
    copied_mb += mb
print(f'archived {len(weights) - len(skipped)}/{len(weights)} checkpoints '
      f'({copied_mb:.1f} MB) into results/artifacts/checkpoints/')
for pth, mb in skipped:
    print(f'   SKIPPED (too large): {pth} at {mb:.1f} MB')

n_records = len(list((out / 'artifacts/audit').glob('*/records.json')))
print('published into results/:')
for p in published:
    print('   ', p)
print(f'\naudit cells with per-molecule records: {n_records}')

# The repo is on ephemeral local disk, so mirror results/ onto Drive as well —
# otherwise a disconnect before the download loses the reports (the audit
# records themselves are already safe under the artifacts symlink).
if DRIVE_DIR:
    mirror = Path(DRIVE_DIR) / 'results'
    if mirror.exists():
        shutil.rmtree(mirror)
    shutil.copytree(out, mirror)
    print('mirrored to', mirror)

print('\nCommit results/ to make this the record of the run.')


## 10. Download the results

Packages **only the run outputs** — reports, figures, checkpoints, audit records,
logs — into a zip written *outside* the staged tree, so the archive can never
contain itself. It refuses to build an archive when the expected outputs are
missing, rather than handing back a plausible-looking but empty zip.

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(REPO_DIR)
STAGE = Path('/content/_molsanity_out')
ZIP_BASE = '/content/molsanity_full_run'  # outside STAGE, so it can't self-include

WANTED = [
    'RESULTS.md', 'BENCHMARK.md', 'BENCHMARK_GT.md', 'BENCHMARK_GT.json',
    'PROGRESS.md', 'LIMITATIONS.md', 'figures', 'logs',
    'artifacts/checkpoints', 'artifacts/audit', 'artifacts/run_manifest.json',
]

if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir(parents=True)

copied = []
for rel in WANTED:
    src = Path(rel)
    if not src.exists():
        continue
    dest = STAGE / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dest) if src.is_dir() else shutil.copy2(src, dest)
    copied.append(rel)

if not any(c in copied for c in ('RESULTS.md', 'figures')):
    raise SystemExit(
        'No run outputs found to package — the sweep produced nothing. '
        f'cwd={os.getcwd()}, found={copied}. Fix the run before downloading.'
    )

zip_path = shutil.make_archive(ZIP_BASE, 'zip', root_dir=STAGE)
print('packaged:', copied)
print('zip:', zip_path, f'({os.path.getsize(zip_path) / 1e6:.1f} MB)')

try:
    from google.colab import files

    files.download(zip_path)
except Exception as exc:
    print('Not on Colab or download blocked — grab it from the file browser:', exc)